# CSV → Parquet con RAPIDS cuDF (Colab GPU) — v2

**Cambios vs v1:**
- **Fix de tipos**: se evita el conflicto `float32` (numpy) vs `Float32` (pandas nullable)
  usando PyArrow directamente con schema explícito.
- **Nueva estructura de carpetas**: `nombre_csv/year=YYYY/` (sin cohortes en subcarpetas).

```
OUTPUT_PATH/
  2000Q1/              ← nombre del CSV sin extensión
    year=2000/
      part-0000.snappy.parquet
    year=2001/
      part-0000.snappy.parquet
  2000Q2/
    year=2000/
    ...
```

## Configuración requerida
1. **Runtime → Change runtime type → GPU** (T4 mínimo)
2. **High RAM** si está disponible

## 1. Montar Google Drive

In [10]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive montado en /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montado en /content/drive


## 2. GPU + Instalación de RAPIDS

In [11]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q --extra-index-url=https://pypi.nvidia.com \
    cudf-cu12 dask-cudf-cu12 dask-cuda rmm-cu12
print("\nRAPIDS instalado.")

NVIDIA L4, 23034 MiB

RAPIDS instalado.


## 3. Cluster CUDA

In [12]:
import os, glob, json, re, time
import cudf
import dask_cudf as dc
import pandas as pd
from dask.distributed import Client
from dask_cuda import LocalCUDACluster
from dask.utils import parse_bytes

gpu_mem_query = !nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits
GPU_MEM_MB = int(gpu_mem_query[0].strip())
RMM_POOL   = f"{int(GPU_MEM_MB * 0.80)}MB"
DEV_LIMIT  = f"{int(GPU_MEM_MB * 0.70)}MB"

print(f"GPU VRAM: {GPU_MEM_MB} MB")
print(f"RMM pool: {RMM_POOL} | Spill threshold: {DEV_LIMIT}")

cluster = LocalCUDACluster(
    CUDA_VISIBLE_DEVICES="0",
    rmm_pool_size=parse_bytes(RMM_POOL),
    device_memory_limit=parse_bytes(DEV_LIMIT),
)
client = Client(cluster)
print(f"Dask dashboard: {client.dashboard_link}")
client

/usr/local/lib/python3.12/dist-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 43333 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:40627
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:43333/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:38905'


GPU VRAM: 23034 MB
RMM pool: 18427MB | Spill threshold: 16123MB


INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:46307 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:46307
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:48696
INFO:distributed.scheduler:Receive client connection: Client-4dd4782b-16a6-11f1-8a14-0242ac1c000c
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:48706


Dask dashboard: http://127.0.0.1:43333/status


Connection method: Cluster object,Cluster type: dask_cuda.LocalCUDACluster
Dashboard: http://127.0.0.1:43333/status,
Dashboard: http://127.0.0.1:43333/status,Workers: 1
Total threads: 1,Total memory: 52.96 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40627,Workers: 0
Dashboard: http://127.0.0.1:43333/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46307,Total threads: 1
Dashboard: http://127.0.0.1:34801/status,Memory: 52.96 GiB
Nanny: tcp://127.0.0.1:38905,


## 4. Parámetros (rutas en Google Drive)

Ajustá `DRIVE_BASE` y las subcarpetas según tu estructura.

In [13]:
DRIVE_BASE   = "/content/drive/MyDrive"
DIR_METADATA = "mortgage-risk/metadata"
DIR_RAW      = "mortgage-risk/"
DIR_PARQUET  = "mortgage-risk/parquet"

GLOSSARY_PATH = f"{DRIVE_BASE}/{DIR_METADATA}/crt_glossary_tables.csv"
INPUT_PATH    = f"{DRIVE_BASE}/{DIR_RAW}"
OUTPUT_PATH   = f"{DRIVE_BASE}/{DIR_PARQUET}"
PROGRESS_FILE = f"{OUTPUT_PATH}/_progress.json"

ORIGINATION_DATE_COL = "Origination_Date"
SEP          = "|"
MAX_COLUMNS  = 110
BLOCKSIZE    = "512MB"

print("Rutas en Drive:")
print(f"  Glosario:  {GLOSSARY_PATH}")
print(f"  CSVs:      {INPUT_PATH}")
print(f"  Parquet:   {OUTPUT_PATH}")
print(f"  Blocksize: {BLOCKSIZE}")

Rutas en Drive:
  Glosario:  /content/drive/MyDrive/mortgage-risk/metadata/crt_glossary_tables.csv
  CSVs:      /content/drive/MyDrive/mortgage-risk/
  Parquet:   /content/drive/MyDrive/mortgage-risk/parquet
  Blocksize: 512MB


## 5. Cargar glosario (pandas)

In [14]:
def spark_safe_name(s):
    return re.sub(r"[\s()]+", "_", s).strip("_").replace("__", "_") or "col"

def _default_columns(n=110):
    return [f"col_{i}" for i in range(n)]

def load_glossary_metadata(glossary_path, max_cols=110):
    g = pd.read_csv(glossary_path, header=0, quoting=1)
    if g.shape[1] < 11:
        return [(c, "ALPHANUMERIC", "") for c in _default_columns(max_cols)]
    out = []
    for _, row in g.iterrows():
        pos_str = str(row.iloc[0]).strip()
        if not pos_str.isdigit():
            continue
        name = (str(row.iloc[1]) or "").strip()
        if not name:
            continue
        gtype = str(row.iloc[9]).strip().upper().replace("-", "")
        maxlen = str(row.iloc[10]).strip() if pd.notna(row.iloc[10]) else ""
        out.append((spark_safe_name(name), gtype, maxlen))
        if len(out) >= max_cols:
            break
    if not out:
        out = [(c, "ALPHANUMERIC", "") for c in _default_columns(max_cols)]
    return out

metadata  = load_glossary_metadata(GLOSSARY_PATH, MAX_COLUMNS)
col_names = [name for name, _, _ in metadata]

print(f"Columnas del glosario: {len(metadata)}")
print(f"Primeras 5: {col_names[:5]}")

Columnas del glosario: 110
Primeras 5: ['Reference_Pool_ID', 'Loan_Identifier', 'Monthly_Reporting_Period', 'Channel', 'Seller_Name']


## 6. Schema y función de transformación (GPU)

**Fix clave**: se usa `np.dtype` nativo en lugar de tipos nullable de pandas.
El `cohort_year` se calcula desde `Origination_Date` pero **NO** se incluye como columna  
en el schema del Parquet — se usa solo para determinar la subcarpeta `year=YYYY/`.

In [15]:
import numpy as np
import cudf
import dask_cudf as dc
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from glob import glob
import os, json, time
from tqdm.notebook import tqdm

# ═══════════════════════════════════════════════════════════════════════════════
# ESQUEMA FINAL: tipos NumPy nativos (NO pandas nullable)
#
# FIX: "ALPHANUMERIC" contiene "NUMERIC" → antes se clasificaban mal como float.
# Orden correcto: 1) ALPHANUMERIC → string, 2) DATE → datetime, 3) NUMERIC → float
#
# FORCE_STRING_COLUMNS: columnas que SIEMPRE deben ser string (identificadores,
# códigos, indicadores Y/N, categorías) aunque el glosario diga NUMERIC.
# ═══════════════════════════════════════════════════════════════════════════════

FORCE_STRING_COLUMNS = frozenset([
    # Identificadores y nombres
    "Reference_Pool_ID", "Loan_Identifier", "Seller_Name", "Servicer_Name",
    "Master_Servicer", "Deal_Name",
    # Indicadores booleanos (Y/N/U)
    "Channel", "First_Time_Home_Buyer_Indicator", "Prepayment_Penalty_Indicator",
    "Interest_Only_Loan_Indicator", "Modification_Flag",
    "Mortgage_Insurance_Cancellation_Indicator", "Servicing_Activity_Indicator",
    "Relocation_Mortgage_Indicator", "Loan_Holdback_Indicator",
    "High_Balance_Loan_Indicator", "ARM_Initial_Fixed-Rate_Period_≤_5_YR_Indicator",
    "ARM_Balloon_Indicator",
    "High_Loan_to_Value_HLTV_Refinance_Option_Indicator",
    "Repurchase_Make_Whole_Proceed", "Payment_Deferral_Modification_Event_Indicator",
    # Categóricas y códigos
    "Loan_Purpose", "Property_Type", "Occupancy_Status", "Property_State",
    "Amortization_Type", "Property_Valuation_Method", "Special_Eligibility_Program",
    "Borrower_Assistance_Plan", "Alternative_Delinquency_Resolution",
    "Mortgage_Insurance_Type",
    # Códigos de estado y geografía (preservar ceros a la izquierda)
    "Zip_Code_Short", "Metropolitan_Statistical_Area_MSA",
    "Current_Loan_Delinquency_Status", "Zero_Balance_Code", "Loan_Payment_History",
    # ARM y estructuras con caracteres especiales
    "ARM_Product_Type", "Index", "ARM_Cap_Structure", "ARM_Plan_Number",
])

def build_final_schema(metadata):
    """Construye esquema con tipos NumPy nativos estrictos."""
    schema = {}
    for _name, _gtype, _maxlen in metadata:
        if _name in FORCE_STRING_COLUMNS:
            schema[_name] = np.dtype("O")  # string forzado
        elif _gtype == "ALPHANUMERIC" or "ALPHA" in _gtype:
            # IMPORTANTE: verificar ALPHANUMERIC ANTES que NUMERIC (evita bug)
            schema[_name] = np.dtype("O")
        elif "DATE" in _gtype:
            schema[_name] = np.dtype("datetime64[ns]")
        elif _gtype == "NUMERIC":
            schema[_name] = np.dtype("float32")
        else:
            schema[_name] = np.dtype("O")
    # cohort_year: columna auxiliar para partición en carpetas (object)
    schema["cohort_year"] = np.dtype("O")
    return schema

FINAL_SCHEMA = build_final_schema(metadata)

# META cuDF explícito
META = cudf.DataFrame({
    k: cudf.Series(dtype=v) for k, v in FINAL_SCHEMA.items()
})

# ═══════════════════════════════════════════════════════════════════════════════
# SCHEMA ARROW EXPLÍCITO — garantiza tipos idénticos en todos los archivos
# ═══════════════════════════════════════════════════════════════════════════════

def build_arrow_schema(final_schema, exclude_cols=None):
    """Convierte FINAL_SCHEMA (numpy dtypes) → pa.Schema Arrow."""
    exclude_cols = exclude_cols or []
    fields = []
    for col, dtype in final_schema.items():
        if col in exclude_cols:
            continue
        if dtype == np.dtype('float32'):
            pa_type = pa.float32()
        elif dtype == np.dtype('float64'):
            pa_type = pa.float64()
        elif dtype == np.dtype('int64'):
            pa_type = pa.int64()
        elif dtype == np.dtype('datetime64[ns]'):
            pa_type = pa.timestamp('ns')
        else:  # object
            pa_type = pa.string()
        fields.append(pa.field(col, pa_type, nullable=True))
    return pa.schema(fields)

# Schema para los datos (sin cohort_year — va en el nombre de la subcarpeta)
ARROW_SCHEMA_DATA = build_arrow_schema(FINAL_SCHEMA, exclude_cols=['cohort_year'])

print(f"FINAL_SCHEMA: {len(FINAL_SCHEMA)} columnas (incl. cohort_year)")
print(f"ARROW_SCHEMA_DATA: {len(ARROW_SCHEMA_DATA)} columnas (sin cohort_year)")

# ═══════════════════════════════════════════════════════════════════════════════
# FUNCIONES AUXILIARES
# ═══════════════════════════════════════════════════════════════════════════════

def _create_null_column(n, dtype):
    """Crea columna cuDF con n nulos y tipo NumPy exacto."""
    if dtype == np.dtype("float32"):
        return cudf.Series(np.full(n, np.nan, dtype=np.float32), dtype=np.float32)
    elif dtype == np.dtype("float64"):
        return cudf.Series(np.full(n, np.nan, dtype=np.float64), dtype=np.float64)
    elif dtype == np.dtype("int64"):
        s = cudf.Series(np.zeros(n, dtype=np.int64), dtype=np.int64)
        s[:] = None
        return s
    elif dtype == np.dtype("datetime64[ns]"):
        return cudf.from_pandas(pd.Series([None] * n, dtype="datetime64[ns]"))
    else:
        return cudf.Series([None] * n, dtype="object")


def _force_numpy_types(df, schema):
    """Fuerza tipos NumPy nativos en todo el DataFrame cuDF."""
    result = cudf.DataFrame()
    for col, target_dtype in schema.items():
        if col not in df.columns:
            result[col] = _create_null_column(len(df), target_dtype)
            continue
        s = df[col]
        actual = s.dtype
        if actual == target_dtype:
            result[col] = s
            continue
        try:
            if target_dtype == np.dtype("float32"):
                if s.dtype == "object":
                    s = s.astype("str").str.strip()
                    s = s.where(s.str.len() > 0, None)
                arr = s.astype("float32").to_numpy()
                result[col] = cudf.Series(arr, dtype=np.float32)
            elif target_dtype == np.dtype("datetime64[ns]"):
                if "datetime" in str(s.dtype):
                    result[col] = cudf.Series(s.to_numpy(), dtype="datetime64[ns]")
                else:
                    temp = cudf.to_datetime(s, errors="coerce")
                    result[col] = cudf.Series(temp.to_numpy(), dtype="datetime64[ns]")
            else:  # object
                if s.dtype == "object":
                    result[col] = s.fillna("").astype("str")
                else:
                    result[col] = s.astype("str").fillna("")
        except Exception as e:
            print(f"Error forzando {col} a {target_dtype}: {e}")
            result[col] = _create_null_column(len(df), target_dtype)
    return result


# ═══════════════════════════════════════════════════════════════════════════════
# TRANSFORMACIÓN PRINCIPAL
# ═══════════════════════════════════════════════════════════════════════════════

def transform_partition(df):
    """Transforma una partición cuDF con tipos NumPy nativos garantizados."""
    if len(df) == 0:
        return cudf.DataFrame({col: cudf.Series(dtype=dtype) for col, dtype in FINAL_SCHEMA.items()})

    # ─── Extraer cohort_year desde Origination_Date ───
    date_col = ORIGINATION_DATE_COL
    if date_col in df.columns:
        raw = df[date_col].fillna("").astype("str").str.strip()
        valid = raw.str.len() >= 6
        year_str = raw.str.slice(2, 6)
        df["cohort_year"] = year_str.where(valid, "unknown")
    else:
        df["cohort_year"] = "unknown"

    # ─── Forzar tipos NumPy nativos ───
    result = _force_numpy_types(df, FINAL_SCHEMA)

    return result


# ═══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN: escribir pandas group → Parquet con schema Arrow forzado
# ═══════════════════════════════════════════════════════════════════════════════

def write_group_to_parquet(pdf_group, out_dir, part_idx, arrow_schema):
    """
    Escribe un grupo pandas como Parquet con schema Arrow explícito.
    Castea cada columna al tipo Arrow correcto para evitar conflictos
    float32 (numpy) vs Float32 (pandas nullable).
    """
    if len(pdf_group) == 0:
        return

    arrays = []
    for field in arrow_schema:
        col = field.name
        pa_type = field.type
        if col not in pdf_group.columns:
            arrays.append(pa.nulls(len(pdf_group), type=pa_type))
        else:
            try:
                arr = pa.array(pdf_group[col], type=pa_type, from_pandas=True)
            except Exception:
                try:
                    arr = pa.array(pdf_group[col].values, type=pa_type, from_pandas=True)
                except Exception:
                    arr = pa.nulls(len(pdf_group), type=pa_type)
            arrays.append(arr)

    table = pa.table(
        dict(zip([f.name for f in arrow_schema], arrays)),
        schema=arrow_schema
    )
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(out_dir, f'part-{part_idx:04d}.snappy.parquet')
    pq.write_table(table, out_file, compression='snappy')


print("Schema y funciones cargadas correctamente ✅")

FINAL_SCHEMA: 111 columnas (incl. cohort_year)
ARROW_SCHEMA_DATA: 110 columnas (sin cohort_year)
Schema y funciones cargadas correctamente ✅


## 7. Procesamiento: CSV → Parquet

**Estructura de salida:**
```
OUTPUT_PATH/
  <nombre_csv>/           ← nombre del CSV sin extensión (ej: 2000Q1)
    year=2000/            ← partición por año
      part-0000.snappy.parquet
    year=2001/
      ...
```

**Fix de tipos:** Se computa cada partición dask individualmente como cuDF DataFrame  
y se escribe con `pyarrow.parquet.write_table` con schema Arrow explícito,  
evitando el conflicto `float32` (numpy) vs `Float32` (pandas nullable).

In [16]:
csv_files = sorted(
    glob(f"{INPUT_PATH}/*.csv") +
    glob(f"{INPUT_PATH}/*.csv.gz")
)

def load_progress():
    try:
        with open(PROGRESS_FILE) as f:
            return set(json.load(f))
    except:
        return set()

def save_progress(done):
    os.makedirs(os.path.dirname(PROGRESS_FILE), exist_ok=True)
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(sorted(done), f)

processed = load_progress()
total = len(csv_files)
pending_files = [f for f in csv_files if os.path.basename(f) not in processed]

print(f"Archivos: {total} | Procesados: {len(processed)} | Pendientes: {len(pending_files)}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ────────────────────────────────────────────────────────────────────────────
# PROCESAMIENTO PRINCIPAL
#
# Por cada CSV:
#   1. Leer con dask-cudf en chunks (GPU)
#   2. Transformar tipos con transform_partition()
#   3. Computar partición a partición (evita acumular en GPU)
#   4. Agrupar por cohort_year y escribir cada año en year=YYYY/
#      con schema Arrow forzado (fix del bug float32 vs Float32)
# ────────────────────────────────────────────────────────────────────────────
if not pending_files:
    print("Nada que procesar.")
else:
    t_start = time.time()

    with tqdm(total=total, initial=len(processed), desc="CSV→Parquet", unit="file") as pbar:

        for csv_file in pending_files:
            fname = os.path.basename(csv_file)
            t1 = time.time()

            # ── Carpeta raíz = nombre del CSV sin extensión ──────────────
            csv_stem = fname.replace('.csv.gz', '').replace('.csv', '')
            csv_out_dir = os.path.join(OUTPUT_PATH, csv_stem)
            os.makedirs(csv_out_dir, exist_ok=True)

            pbar.set_postfix_str(f'{fname}')

            # ── Leer CSV con dask-cudf ───────────────────────────────────
            is_gz = csv_file.endswith('.gz')
            ddf = dc.read_csv(
                csv_file,
                sep=SEP,
                header=None,
                names=col_names,
                dtype='str',
                na_values=[''],
                blocksize=None if is_gz else BLOCKSIZE,
            )

            # ── Transformar tipos en GPU ─────────────────────────────────
            ddf = ddf.map_partitions(transform_partition, meta=META)

            # ── Escribir partición a partición con schema Arrow forzado ──
            n_parts = ddf.npartitions
            global_part_idx = 0

            for i in range(n_parts):
                # Computar una sola partición a la vez (gestiona VRAM)
                cudf_part = ddf.get_partition(i).compute()

                if len(cudf_part) == 0:
                    del cudf_part
                    continue

                # Convertir a pandas para groupby
                pdf_part = cudf_part.to_pandas()
                del cudf_part  # liberar VRAM inmediatamente

                # Agrupar por cohort_year → each year gets its own subfolder
                for year_val, pdf_group in pdf_part.groupby('cohort_year', sort=False):
                    # Eliminar columna de partición (va en el path, no en los datos)
                    pdf_data = pdf_group.drop(columns=['cohort_year'])

                    year_dir = os.path.join(csv_out_dir, f'year={year_val}')

                    write_group_to_parquet(
                        pdf_data,
                        out_dir=year_dir,
                        part_idx=global_part_idx,
                        arrow_schema=ARROW_SCHEMA_DATA,  # sin cohort_year
                    )

                del pdf_part
                global_part_idx += 1

            # ── Guardar progreso ─────────────────────────────────────────
            processed.add(fname)
            save_progress(processed)

            elapsed = time.time() - t1
            year_dirs = [d for d in os.listdir(csv_out_dir) if d.startswith('year=')]
            pbar.set_postfix({
                'archivo': fname[:20],
                'tiempo': f'{elapsed:.1f}s',
                'años': len(year_dirs)
            })
            pbar.update(1)

    print(f"\n✅ Completado en {(time.time()-t_start)/60:.1f} minutos")
    print(f"   Output: {OUTPUT_PATH}")

Archivos: 100 | Procesados: 82 | Pendientes: 18


CSV→Parquet:  82%|########2 | 82/100 [00:00<?, ?file/s]


✅ Completado en 99.3 minutos
   Output: /content/drive/MyDrive/mortgage-risk/parquet


## 8. Verificación de salida

In [17]:
import subprocess

result = subprocess.run(["du", "-sh", OUTPUT_PATH], capture_output=True, text=True)
print(f"Tamaño total Parquet: {result.stdout.strip()}")

print("\nEstructura de carpetas:")
csv_dirs = sorted([d for d in os.listdir(OUTPUT_PATH) if os.path.isdir(os.path.join(OUTPUT_PATH, d)) and not d.startswith('_')])
print(f"  Total CSVs procesados: {len(csv_dirs)}")

for csv_dir in csv_dirs:
    csv_path = os.path.join(OUTPUT_PATH, csv_dir)
    year_dirs = sorted([d for d in os.listdir(csv_path) if d.startswith('year=')])
    print(f"\n  📁 {csv_dir}/ → {len(year_dirs)} años")
    for yd in year_dirs[:5]:
        parts = glob(os.path.join(csv_path, yd, '*.parquet'))
        print(f"       {yd}/  ({len(parts)} archivos parquet)")
    if len(year_dirs) > 5:
        print(f"       ... y {len(year_dirs)-5} años más")

# Verificar tipos leyendo un archivo muestra
print("\n📊 Verificación de tipos (muestra del primer parquet):")
sample_files = glob(f"{OUTPUT_PATH}/**/year=*/*.parquet", recursive=True)
if sample_files:
    sample_table = pq.read_table(sample_files[0])
    print(f"   Archivo: {sample_files[0]}")
    print(f"   Filas: {len(sample_table):,}")
    for field in sample_table.schema:
        print(f"   {field.name}: {field.type}")
else:
    print("   No se encontraron archivos parquet")

Tamaño total Parquet: 24G	/content/drive/MyDrive/mortgage-risk/parquet

Estructura de carpetas:
  Total CSVs procesados: 99

  📁 2000Q1/ → 2 años
       year=1999/  (10 archivos parquet)
       year=2000/  (10 archivos parquet)

  📁 2000Q2/ → 2 años
       year=1999/  (9 archivos parquet)
       year=2000/  (9 archivos parquet)

  📁 2000Q3/ → 2 años
       year=1999/  (9 archivos parquet)
       year=2000/  (9 archivos parquet)

  📁 2000Q4/ → 2 años
       year=1999/  (11 archivos parquet)
       year=2000/  (11 archivos parquet)

  📁 2001Q1/ → 3 años
       year=1999/  (17 archivos parquet)
       year=2000/  (17 archivos parquet)
       year=2001/  (17 archivos parquet)

  📁 2001Q2/ → 3 años
       year=1999/  (33 archivos parquet)
       year=2000/  (33 archivos parquet)
       year=2001/  (33 archivos parquet)

  📁 2001Q3/ → 3 años
       year=1999/  (30 archivos parquet)
       year=2000/  (30 archivos parquet)
       year=2001/  (30 archivos parquet)

  📁 2002Q1/ → 4 años
       

## 9. Cerrar cluster

In [18]:
client.close()
cluster.close()
print("Cluster CUDA cerrado.")

INFO:distributed.scheduler:Remove client Client-4dd4782b-16a6-11f1-8a14-0242ac1c000c
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:48706; closing.
INFO:distributed.scheduler:Remove client Client-4dd4782b-16a6-11f1-8a14-0242ac1c000c
INFO:distributed.scheduler:Close client connection: Client-4dd4782b-16a6-11f1-8a14-0242ac1c000c
INFO:distributed.scheduler:Retire worker addresses (stimulus_id='retire-workers-1772509857.506811') (0,)
INFO:distributed.nanny:Closing Nanny at 'tcp://127.0.0.1:38905'. Reason: nanny-close
INFO:distributed.nanny:Nanny asking worker to close. Reason: nanny-close
INFO:distributed.core:Received 'close-stream' from tcp://127.0.0.1:48696; closing.
INFO:distributed.scheduler:Remove worker addr: tcp://127.0.0.1:46307 name: 0 (stimulus_id='handle-worker-cleanup-1772509857.511846')
INFO:distributed.scheduler:Lost all workers
INFO:distributed.nanny:Nanny at 'tcp://127.0.0.1:38905' closed.
INFO:distributed.scheduler:Closing scheduler. Reason: unknown
IN

Cluster CUDA cerrado.
